# *Voces Magicae* in the Residual Stream — CROSS-FAMILY variant

Derived from the v6 notebook; **every science cell is identical** — only the model-loading is changed, so
results are directly comparable to the Qwen runs. **Pick a model in the config cell (Cell 2).** The point is a
*different tokenizer + different pretraining* (Gemma, Llama, Mistral) to test whether H1 and the Greek finding
generalize or were single-tokenizer artifacts.

- **Gated models (Gemma/Llama)** need an HF token: accept the license on the model's HF page, then add
  `HF_TOKEN` to Colab **Secrets** (🔑 in the sidebar). **Mistral and Qwen need no token.**
- **Llama-405B will not run here** (even 4-bit ≈ 200 GB). Use Gemma-2-9B or Llama-3.1-8B — a different
  tokenizer is the test, not raw scale.
- Outputs are tagged by model (`voces_xfam-<model>_results.json`) so multiple models don't overwrite each
  other. Send the JSON back and it can be compared against the Qwen decider.


# *Voces Magicae* in the Residual Stream — MVP Pilot

**Handoff provenance.** Designed by an Opus-4.8 instance on claude.ai (API-only, white-box-blind),
specced in `grimoire-interpretability/SKILL.md` + `references/voces-contrast-study.md`, built into this
runnable notebook by an Opus-4.8 Claude Code instance, to be executed on Colab GPU. Results flow back up
the chain.

**What this is.** The MVP cut of the spec (§11, steps 1–5; H1 separability + H2 name-likeness), built around
the **one decisive bet (§2a)**: the 2×2 **(pure-asemic vs theonym-bearing) × (Latin vs Greek script)**, with
**surprisal and graded theonymic contamination (T) regressed out**.

**What this is NOT.** The ≥7B headline science *at scale*, run to publication power. This is a pilot on one
model. Read every number as provisional.

---

### The four falsifiers, named in advance (so the apparatus can't launder a result)

1. **Name-likeness is entirely the theonym subset** (T explains it; pure-asemic voces are not name-like)
   → finding shrinks to "the model reads corrupt god-names."
2. **Signature dies under Greek-script migration** → it was typographic string-cache, not abstract form.
3. *(steering — H4, out of MVP scope; not tested here).*
4. **Separability is surprisal in disguise** (regress frequency out and H1 falls to chance)
   → "distinct from unseen shadows" was tautological.

This notebook reports evidence on **#1, #2, #4** *flat* — including when they kill the headline. The
match-quality figures (§Cell 8) are printed **before** the headline, because if the token-match control isn't
actually matched, every downstream number is confounded.

### Known limitations of THIS pilot (read before trusting anything)
- **Corpus is curated from author/model knowledge of the PGM, not parsed from a verified Betz edition.**
  Theonym tags and structure tags are hand-assigned. This is a pilot stimulus set, not a provenance-clean corpus.
- **Token-rarity proxy = the model's own per-string surprisal** (we have no external pretraining-corpus
  frequency table here). The §4 spec allows the intrinsic surprisal proxy; we lean on it.
- **Greek script is generated by an algorithmic Betz→Greek transliterator**, not sourced from Preisendanz.
  Good enough to test byte-string vs abstract-form; not an editorial-variance study.
- **Scale (falsifier-caveat #0).** Default model is chosen by available VRAM. Below a T4 it falls to 3B/1.5B
  — *plumbing tier*, label results exploratory. This is a pilot, not the ≥7B-at-power headline.
- **Quantization (falsifier-caveat #2, sits next to scale).** A 16 GB T4 reaches a 7B only in **4-bit**, and
  4-bit quantization *perturbs the residual stream* — quantization noise can blur or shift the very directions
  this study measures. Almost certainly fine for the **coarse 2×2** (name-likeness is a large effect if it
  exists at all), but if the headline comes out of a 4-bit model the honest label is **"results may be
  sensitive to quantization; fp16 on a larger GPU is the confirmation pass."** The notebook prints this loudly
  whenever it loads 4-bit, records `quantized: true` in the results JSON, and repeats it on the verdict board —
  so quantization never becomes an invisible confound the way tokenization almost did. To kill the caveat, run
  fp16 (set `LOAD_4BIT=False` on a ≥24 GB GPU) and confirm the 2×2 holds.



## Cell 1 — Install & sanity


In [ ]:
%pip -q install -U transformers accelerate bitsandbytes scikit-learn scipy huggingface_hub
import torch, transformers, sklearn, scipy, numpy as np, platform
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > GPU."
print("GPU", torch.cuda.get_device_name(0), "|", round(torch.cuda.get_device_properties(0).total_memory/1e9,1), "GB")


## Cell 2 — Config: pick the model from available VRAM

Default policy reaches the spec's **≥7B science tier** when the GPU allows (T4/L4/A100 → Qwen2.5-7B in 4-bit),
and degrades gracefully to 3B (fp16) or 1.5B (fp16) on smaller GPUs. Qwen2.5 is chosen for genuine **Greek +
Latin** coverage (load-bearing for the script-migration arm) and a modern BPE tokenizer.

**Override freely** by editing `MODEL_NAME` / `LOAD_4BIT` below — e.g. if you have an A100, set
`MODEL_NAME="Qwen/Qwen2.5-7B"; LOAD_4BIT=False` for clean (un-quantized) activations.


In [ ]:
import torch, numpy as np, random
SEED=0; np.random.seed(SEED); random.seed(SEED); torch.manual_seed(SEED)

# ===========================================================================
# CROSS-FAMILY TEST -- the scientific point is a DIFFERENT tokenizer + different
# pretraining than Qwen, to see whether H1 (texture recognition) and the Greek
# finding generalize, or were Qwen/single-tokenizer artifacts.
#
#   Open, no token:   "Qwen/Qwen2.5-7B"             (in-family baseline)
#                     "mistralai/Mistral-7B-v0.3"   (cross-family, ZERO friction)
#   GATED (need an HF token + accept the license on the model's HF page first):
#                     "google/gemma-2-9b"   "google/gemma-2-2b"
#                     "meta-llama/Llama-3.1-8B"
#
#   Llama-3.1-405B does NOT fit Colab (even 4-bit ~200 GB -> multi-GPU host).
#   The 8B is the runnable Llama; 70B needs an A100-80G (Colab Pro+).
# ===========================================================================
MODEL_NAME = "google/gemma-2-9b"   # <-- CHANGE ME to test a model
LOAD_4BIT  = True                  # 4-bit fits a 16GB T4; False = fp16 (needs a big GPU)

# HF auth for GATED models (Gemma / Llama). Add HF_TOKEN to Colab Secrets (the key
# icon in the left sidebar) and accept the model's license page; or paste below.
# Not needed for Qwen / Mistral.
import os
HF_TOKEN = os.environ.get("HF_TOKEN", "")
try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or (userdata.get("HF_TOKEN") or "")
except Exception:
    pass
# HF_TOKEN = "hf_..."   # <- or paste your token here
if HF_TOKEN:
    from huggingface_hub import login; login(HF_TOKEN); print("HF: logged in")
else:
    print("HF: no token (OK for Qwen/Mistral; Gemma/Llama will 401 -> add HF_TOKEN to Secrets + accept the license)")

TAG = MODEL_NAME.split("/")[-1]
VERSION = "xfam-" + TAG            # output files are tagged by model so runs don't overwrite
TIER = ("4bit" if LOAD_4BIT else "fp16") + "/" + TAG

# --- guard: BASE variants only. The Qwen baseline was Qwen2.5-3B (BASE). An -it / -Instruct /
#     -chat model would confound "different FAMILY" with "different POST-TRAINING". ---
if any(k in MODEL_NAME.lower() for k in ["-it", "instruct", "-chat", "-sft", "-dpo"]):
    print("!!", "="*60)
    print("!! WARNING: this looks like an INSTRUCT/CHAT model. The baseline is BASE (Qwen2.5-3B).")
    print("!! Use the BASE variant (e.g. google/gemma-2-9b, NOT -9b-it) or you confound family")
    print("!! with post-training. Change MODEL_NAME unless this is deliberate.")
    print("!!", "="*60)

# --- pre-flight gated-access check: fail SOFT on a still-pending gate (e.g. Llama in Meta's queue),
#     so it never 403-crashes a run. Gemma is typically instant; Llama-3.1 can take hours. ---
ACCESS_OK = True
try:
    from huggingface_hub import auth_check
except Exception:
    auth_check = None
if auth_check:
    try:
        auth_check(MODEL_NAME)
        print("access OK:", MODEL_NAME)
    except Exception as e:
        ACCESS_OK = False
        print("!! ACCESS NOT GRANTED for", MODEL_NAME, "->", type(e).__name__)
        print("   Gated & still pending? Wait for approval, or set MODEL_NAME to an approved model")
        print("   (Gemma-2-9b ready now; Mistral needs no gate) and re-run. The load cell will skip cleanly.")
print("MODEL:", MODEL_NAME, "| 4-bit:", LOAD_4BIT, "| outputs ->", "voces_" + VERSION + "_results.json")


## Cell 3 — Load model & tokenizer


In [ ]:
if not ACCESS_OK:
    raise SystemExit("No access to " + MODEL_NAME + " yet (gated/pending). Change MODEL_NAME to an "
                     "approved model and re-run all — nothing below this cell ran, so no partial output.")
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None: tok.pad_token = tok.eos_token
kw = dict(torch_dtype=torch.float16, device_map="auto",
          output_hidden_states=True, low_cpu_mem_usage=True)
if "gemma-2" in MODEL_NAME.lower():
    kw["attn_implementation"] = "eager"   # recommended for Gemma-2 (logit soft-capping)
if LOAD_4BIT:
    kw["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **kw).eval()
N_LAYERS = model.config.num_hidden_layers
D_MODEL  = model.config.hidden_size
DEV = next(model.parameters()).device
print("loaded:", MODEL_NAME, "| layers:", N_LAYERS, "| d_model:", D_MODEL, "| device:", DEV)


## Cell 4 — Corpus: attested voces (Latin), theonym lexicon, Betz→Greek transliterator

The corpus is hand-curated PGM material (see limitations up top). Each entry carries a **family** (for
GroupKFold — the probe must generalize *across* families, not memorize strings), a **theonym_bearing** flag,
and **structure tags**. The Greek-script form is generated by an algorithmic transliterator so the Latin and
Greek cohorts are the *same strings* at different byte-level encodings — exactly the contrast falsifier #2 needs.


In [ ]:
# --- attested voces: (LATIN, family, theonym_bearing, [structure_tags]) ---
VOCES_RAW = [
    ("ABLANATHANALBA",        "palindrome", False, ["palindrome"]),
    ("AKRAMMACHAMAREI",       "akram",      False, ["reduplication"]),
    ("SESENGENBARPHARANGES",  "sesengen",   False, ["long"]),
    ("SEMESILAM",             "semes",      False, ["solar"]),
    ("SEMESEILAMPS",          "semes",      False, ["solar"]),
    ("MASKELLI",              "maskelli",   False, ["formula"]),
    ("MASKELLO",              "maskelli",   False, ["formula"]),
    ("NEBOUTOSOUALETH",       "nebou",      False, ["long"]),
    ("DAMNAMENEUS",           "ephesia",    False, ["ephesia"]),
    ("ASKION",                "ephesia",    False, ["ephesia"]),
    ("KATASKION",             "ephesia",    False, ["ephesia","reduplication"]),
    ("LIXTETRAX",             "ephesia",    False, ["ephesia"]),
    ("AISION",                "ephesia",    False, ["ephesia"]),
    ("BAINCHOOOCH",           "baincho",    False, ["vowel_stacking"]),
    ("THOBARRABAU",           "thobarra",   False, ["palindrome"]),
    ("CHABRACH",              "chabrach",   False, []),
    ("PHNESCHER",             "chabrach",   False, []),
    ("BORKA",                 "borka",      False, []),
    ("PHORBA",                "phorba",     False, ["reduplication"]),
    ("AROURARELYOTH",         "arour",      False, ["theonym_suffix"]),
    ("KMEPH",                 "kmeph",      False, []),
    ("THENOR",                "thenor",     False, []),
    ("AEEIOYO",               "vowel",      False, ["vowel_string"]),
    ("IEOUAEOI",              "vowel",      False, ["vowel_string"]),
    ("OREOBAZAGRA",           "oreoba",     False, []),
    ("PROTOPHANES",           "proto",      False, []),
    # ---- theonym-bearing ----
    ("ARBATHIAO",             "iao",        True,  ["theonym_fused"]),
    ("PHNOUKENTABAOTH",       "baoth",      True,  ["theonym_fused"]),
    ("MARMARAOTH",            "baoth",      True,  ["theonym_suffix"]),
    ("IAO",                   "iao",        True,  ["pure_theonym","vowel"]),
    ("SABAOTH",               "baoth",      True,  ["pure_theonym"]),
    ("ADONAI",                "adonai",     True,  ["pure_theonym"]),
    ("ABRASAX",               "abrasax",    True,  ["pure_theonym"]),
    ("ABRAXAS",               "abrasax",    True,  ["pure_theonym"]),
    ("ELOAI",                 "eloai",      True,  ["pure_theonym"]),
    ("IAOSABAOTHADONAI",      "concat",     True,  ["theonym_concat"]),
    ("ERESHKIGAL",            "ereshkigal", True,  ["real_theonym_foreign"]),
    ("MICHAEL",               "angel",      True,  ["pure_theonym"]),
    # ---- n-bump (2026-06-24): more attested PGM voces, mostly pure-asemic, to power the low-T subset ----
    ("NEPHERIERI",            "nepheri",    False, ["aphrodite_vox"]),
    ("AKTIOPHI",              "aktio",      False, ["hekate_vox"]),
    ("ORORIOUTH",            "oror",       False, ["uterine_vox"]),
    ("BAKAXICHYCH",           "bakax",      False, ["reduplication"]),
    ("ABERAMENTHOOU",         "aberam",     False, ["long","lion_vox"]),
    ("LERTHEXANAX",           "lerthex",    False, ["long"]),
    ("SOROORMERPHERGAR",      "soroor",     False, ["long"]),
    ("PATATHNAX",             "patath",     False, []),
    ("IARBATHA",              "iarbatha",   False, ["reduplication"]),
    ("SANKANTHARA",           "sankan",     False, []),
    ("PSINOTHER",             "psino",      False, ["theonym_suffix"]),
    ("CHTHETHONI",            "chtheth",    False, []),
    ("MENEBAINCHUCH",         "menebai",    False, ["vowel_stacking"]),
    ("BARZAN",                "barzan",     False, []),
    ("SISISRO",               "sisis",      False, ["reduplication"]),
    ("THENOB",                "thenob",     False, []),
    ("ARSENOPHRE",            "arseno",     False, []),
    ("BACHUCH",               "bachuch",    False, ["reduplication"]),
    ("PHORPHORBA",            "phorba",     False, ["reduplication"]),
    ("OREOBAZAGRA",           "oreoba",     False, []),
    # ---- more theonym-bearing ----
    ("PAKERBETH",             "seth",       True,  ["theonym_fused"]),
    ("BOLCHOSETH",            "seth",       True,  ["theonym_fused"]),
    ("THALAMABAOTH",          "baoth",      True,  ["theonym_suffix"]),
    ("MARMARAUOTH",           "baoth",      True,  ["theonym_suffix","attested_greek"]),
    ("BARBARIOTH",            "oth",        True,  ["theonym_suffix"]),
    # ---- VERIFIED from Digital Ambler / Betz: PGM VII.795-845 zodiacal names (authentic Greek script) ----
    ("HARMONTHARTHOCHE",      "zodiac",     False, ["attested_greek","pgm_vii"]),
    ("NEOPHOXOTHA",           "zodiac",     False, ["attested_greek","pgm_vii"]),
    ("ARISTANABAZAO",         "zodiac",     False, ["attested_greek","pgm_vii"]),
    ("PCHORBAZANACHU",        "zodiac",     False, ["attested_greek","pgm_vii"]),
    ("ZALAMOIRLALITH",        "zodiac",     False, ["attested_greek","pgm_vii"]),
    ("EILESILARMU",           "zodiac",     False, ["attested_greek","pgm_vii"]),
    ("TANTINURACHTH",         "zodiac",     False, ["attested_greek","pgm_vii"]),
    ("CHORCHORNATHI",         "zodiac",     False, ["attested_greek","pgm_vii","reduplication"]),
    ("PHANTHENPHYPHLIA",      "zodiac",     False, ["attested_greek","pgm_vii"]),
    ("AZAZAEISTHAILICH",      "zodiac",     False, ["attested_greek","pgm_vii"]),
    ("MENNYTHYTH",            "zodiac",     False, ["attested_greek","pgm_vii"]),
    ("SERYCHARRALMIO",        "zodiac",     False, ["attested_greek","pgm_vii"]),
    ("AKHEBUKROM",            "solar",      False, ["attested_greek","pgm_xiii"]),
]

# --- theonym ROOTS for graded contamination scoring T (substring coverage) ---
THEONYM_ROOTS = ["IAO","SABAOTH","BAOTH","AOTH","ADONAI","ELOAI","ELOI","ABRASAX","ABRAXAS","ABRA",
                 "MICHA","ERESHKIGAL","SABA","ADON","OTH","IAE","IEOU","SETH","HOR","OSIR","ISIS","AMOUN","THOTH"]

# --- Betz Latin -> uppercase Greek transliterator (digraphs first) ---
_DIGRAPHS = [("TH","Θ"),("PH","Φ"),("CH","Χ"),("PS","Ψ"),("NG","ΝΓ")]
_SINGLES  = {"A":"Α","B":"Β","G":"Γ","D":"Δ","E":"Ε","Z":"Ζ","H":"Η","I":"Ι","K":"Κ","L":"Λ",
             "M":"Μ","N":"Ν","X":"Ξ","O":"Ο","P":"Π","R":"Ρ","S":"Σ","T":"Τ","U":"Υ","Y":"Υ","W":"Ω"}
def to_greek(s):
    s = s.upper()
    for a,b in _DIGRAPHS: s = s.replace(a,b)
    return "".join(_SINGLES.get(c, c) for c in s)

# --- AUTHENTIC Greek spellings (verbatim from Betz/Digital Ambler) override the algorithmic transliterator.
#     PGM VII.795-845 zodiacal names + PGM XIII solar name. Strongest form of Falsifier#2: real Greek bytes. ---
ATTESTED_GREEK = {
    "HARMONTHARTHOCHE":"ΑΡΜΟΝΘΑΡΘΩΧΕ", "NEOPHOXOTHA":"ΝΕΟΦΟΞΩΘΑ", "ARISTANABAZAO":"ΑΡΙΣΤΑΝΑΒΑΖΑΩ",
    "PCHORBAZANACHU":"ΠΧΟΡΒΑΖΑΝΑΧΟΥ", "ZALAMOIRLALITH":"ΖΑΛΑΜΟΙΡΛΑΛΙΘ", "EILESILARMU":"ΕΙΛΕΣΙΛΑΡΜΟΥ",
    "TANTINURACHTH":"ΤΑΝΤΙΝΟΥΡΑΧΘ", "CHORCHORNATHI":"ΧΟΡΧΟΡΝΑΘΙ", "PHANTHENPHYPHLIA":"ΦΑΝΘΕΝΦΥΦΛΙΑ",
    "AZAZAEISTHAILICH":"ΑΖΑΖΑΕΙΣΘΑΙΛΙΧ", "MENNYTHYTH":"ΜΕΝΝΥΘΥΘ", "SERYCHARRALMIO":"ΣΕΡΥΧΑΡΡΑΛΜΙΩ",
    "AKHEBUKROM":"ΑΧΕΒΥΚΡΩΜ", "MARMARAUOTH":"ΜΑΡΜΑΡΑΥΩΘ",
}
SOURCE = {**{k:"PGM VII.795-845 (Betz; Digital Ambler 2018)" for k in
            ["HARMONTHARTHOCHE","NEOPHOXOTHA","ARISTANABAZAO","PCHORBAZANACHU","ZALAMOIRLALITH",
             "EILESILARMU","TANTINURACHTH","CHORCHORNATHI","PHANTHENPHYPHLIA","AZAZAEISTHAILICH",
             "MENNYTHYTH","SERYCHARRALMIO"]},
          "AKHEBUKROM":"PGM XIII (Digital Ambler 2023)", "MARMARAUOTH":"PGM IV.930-1114 (Betz)"}

def contamination_T(s):
    s = s.upper(); covered = [False]*len(s)
    for root in THEONYM_ROOTS:
        start = 0
        while True:
            i = s.find(root, start)
            if i < 0: break
            for j in range(i, i+len(root)): covered[j] = True
            start = i+1
    return sum(covered)/max(1,len(s))

print("n voces:", len(VOCES_RAW),
      "| theonym-bearing:", sum(v[2] for v in VOCES_RAW),
      "| pure-asemic:", sum(not v[2] for v in VOCES_RAW))
print("transliteration check  ABLANATHANALBA ->", to_greek("ABLANATHANALBA"))
print("contamination examples:",
      [(v[0], round(contamination_T(v[0]),2)) for v in VOCES_RAW[:4]+VOCES_RAW[-6:]])



## Cell 5 — Anchor cohorts: names, words, random (Latin + Greek)

`v_name_control` (real proper/divine names) and `v_word_baseline` (common words) are the geometric anchors for
H2; `v_random` is the noise floor. Each is generated in **both scripts** so the script-migration arm has matched
anchors on both sides.


In [ ]:
NAMES_LAT = ["AGAMEMNON","ODYSSEUS","ACHILLEUS","PENELOPE","ALEXANDROS","SOKRATES","ARISTOTELES",
             "HERAKLES","PERSEPHONE","APHRODITE","POSEIDON","ARTEMIS","DIONYSOS","ASKLEPIOS",
             "HEPHAISTOS","KLEOPATRA"]
WORDS_LAT = ["THALASSA","OIKOS","ARTOS","LITHOS","POTAMOS","HEMERA","NYX","HIPPOS","DENDRON",
             "ANTHROPOS","PHILIA","CHRONOS","OURANOS","GAIA"]
_CONS = list("BGDZKLMNPRSTFCH"); _VOW = list("AEIOUY")
def rand_consonant_string(n):
    return "".join(random.choice(_CONS) for _ in range(n))
RANDOM_LAT = [rand_consonant_string(random.randint(5,11)) for _ in range(28)]

cohorts_latin = {
    "name":   NAMES_LAT,
    "word":   WORDS_LAT,
    "random": RANDOM_LAT,
}
cohorts_greek = {k:[to_greek(x) for x in v] for k,v in cohorts_latin.items()}
print({k:len(v) for k,v in cohorts_latin.items()})
print("name greek sample:", cohorts_greek["name"][:3])



## Cell 6 — Tokenizer-matched controls (`v_token_match`) — the load-bearing control

BPE shreds `ABLANATHANALBA` into rare-subword soup; a probe that "separates voces from words" may just be
reading token rarity. So for every attested vox we **search** for a phonotactically-plausible nonsense string
whose **(token-count, mean per-string surprisal)** match the target within tolerance, rejecting anything that
looks like a real word. Surprisal stands in for token-rarity (no external freq table here) and is computed
under *this* model. `v_scrambled` (exact character multiset, order destroyed) is generated alongside.

Surprisal is computed first (Cell 6a) because the match search needs it.


In [ ]:
import torch, numpy as np

@torch.no_grad()
def string_surprisal(s):
    ids = tok(s, return_tensors="pt", add_special_tokens=True).input_ids.to(DEV)
    if ids.shape[1] < 2: return float("nan"), ids.shape[1]
    out = model(ids)
    logits = out.logits[0, :-1].float()
    tgt = ids[0, 1:]
    logp = torch.log_softmax(logits, dim=-1)[range(tgt.shape[0]), tgt]
    return float((-logp).mean()), ids.shape[1]

def n_tokens(s):
    return len(tok(s, add_special_tokens=False).input_ids)

# surprisal + token count for every attested vox (both scripts)
VOCES = []
for lat, fam, theo, tags in VOCES_RAW:
    gr = ATTESTED_GREEK.get(lat, to_greek(lat))   # authentic Greek where we have it, else algorithmic
    s_lat, nt_lat = string_surprisal(lat)
    s_gr,  nt_gr  = string_surprisal(gr)
    VOCES.append(dict(latin=lat, greek=gr, family=fam, theonym=theo, tags=tags,
                      source=SOURCE.get(lat,"curated-from-PGM-knowledge"),
                      T=contamination_T(lat),
                      surp_lat=s_lat, ntok_lat=nt_lat, surp_gr=s_gr, ntok_gr=nt_gr))
print("computed surprisal for", len(VOCES), "voces |",
      sum(v['source']!='curated-from-PGM-knowledge' for v in VOCES), "with explicit PGM citation |",
      sum('attested_greek' in v['tags'] for v in VOCES), "with authentic Greek script")
print("sample:", {k:VOCES[0][k] for k in ['latin','ntok_lat','surp_lat','T']})



In [ ]:
# --- phonotactic sampler + matched search for v_token_match (Latin) ---
SYL_C = ["B","G","D","TH","K","L","M","N","P","R","S","T","PH","CH"]
SYL_V = ["A","E","I","O","U","A","E","O"]   # weighted toward common vowels
def gen_pseudoword(target_len):
    out = ""
    while len(out) < target_len:
        out += random.choice(SYL_C) + random.choice(SYL_V)
    return out[:target_len].upper()

_REAL = set(w.upper() for w in (NAMES_LAT+WORDS_LAT+
            ["TABLE","WATER","STONE","LIGHT","HORSE","RIVER","HOUSE","BREAD","FIELD","NIGHT"]))

def match_token_control(target_latin, target_ntok, target_surp, n_try=400,
                        tok_tol=0, surp_tol=1.2):
    best=None; best_d=1e9
    L=len(target_latin)
    for _ in range(n_try):
        cand = gen_pseudoword(random.randint(max(3,L-1), L+1))
        if cand in _REAL: continue
        nt = n_tokens(cand)
        if abs(nt-target_ntok) > tok_tol+1: continue   # soft: within ~1 token
        sp,_ = string_surprisal(cand)
        d = abs(nt-target_ntok)*2.0 + abs(sp-target_surp)
        if d < best_d:
            best_d, best = d, (cand, nt, sp)
        if abs(nt-target_ntok)<=tok_tol and abs(sp-target_surp)<=surp_tol:
            return cand, nt, sp
    return best if best else (gen_pseudoword(L), None, None)

for v in VOCES:
    cand, nt, sp = match_token_control(v["latin"], v["ntok_lat"], v["surp_lat"])
    v["tokmatch_lat"] = cand
    v["tokmatch_ntok_lat"] = nt if nt is not None else n_tokens(cand)
    v["tokmatch_surp_lat"] = sp if sp is not None else string_surprisal(cand)[0]
    v["tokmatch_gr"] = to_greek(cand)
    g_s, g_nt = string_surprisal(v["tokmatch_gr"])
    v["tokmatch_surp_gr"], v["tokmatch_ntok_gr"] = g_s, g_nt
    # scrambled (latin): exact multiset, order destroyed
    chars = list(v["latin"]); random.shuffle(chars)
    v["scrambled_lat"] = "".join(chars)
    v["scrambled_gr"]  = to_greek(v["scrambled_lat"])
print("built token-match + scrambled controls for", len(VOCES), "voces")
print("example:", VOCES[0]["latin"], "->", VOCES[0]["tokmatch_lat"],
      "| ntok", VOCES[0]["ntok_lat"], "vs", VOCES[0]["tokmatch_ntok_lat"])



## Cell 7 — Extraction harness: residual stream per string, all layers

Carrier frame is **identical for every string** (`The string ___ is written on the page.`) so the frame can't
separate classes — only the string can. We mean-pool the residual stream over **the string's own token span**
(not the whole prompt), at every layer, capturing the state the model represents the string *in*. Both
last-token and mean-pooled reps are stored.


In [ ]:
import torch, numpy as np
FRAME_PRE = "The string "
FRAME_POST = " is written on the page."

@torch.no_grad()
def extract_reps(s):
    pre_ids  = tok(FRAME_PRE, add_special_tokens=True).input_ids
    full_ids = tok(FRAME_PRE + s, add_special_tokens=True).input_ids
    start = len(pre_ids)
    end   = len(full_ids)                      # string span = [start, end)
    framed = FRAME_PRE + s + FRAME_POST
    ids = tok(framed, return_tensors="pt", add_special_tokens=True).input_ids.to(DEV)
    end = min(end, ids.shape[1])
    if end <= start: start = max(0, end-1)
    hs = model(ids).hidden_states            # tuple len N_LAYERS+1, each [1,seq,d]
    mean_pool = np.stack([h[0, start:end, :].float().mean(0).cpu().numpy() for h in hs])  # [L+1, d]
    last_tok  = np.stack([h[0, end-1, :].float().cpu().numpy() for h in hs])              # [L+1, d]
    return mean_pool, last_tok

def build_matrix(strings):
    M=[]
    for s in strings:
        mp,_ = extract_reps(s); M.append(mp)
    return np.stack(M)   # [n, L+1, d]

print("extraction harness ready. span-check on a sample:")
_pre = tok(FRAME_PRE, add_special_tokens=True).input_ids
_full = tok(FRAME_PRE+"ABLANATHANALBA", add_special_tokens=True).input_ids
print("  prefix tokens:", len(_pre), "| string span tokens:", len(_full)-len(_pre))



In [ ]:
# --- run extraction for all cohorts, both scripts ---  (the slow cell)
import time; t0=time.time()
def cohort_strings(script):
    key = "latin" if script=="latin" else "greek"
    attested  = [v[key] for v in VOCES]
    tokmatch  = [v["tokmatch_lat" if script=="latin" else "tokmatch_gr"] for v in VOCES]
    scrambled = [v["scrambled_lat" if script=="latin" else "scrambled_gr"] for v in VOCES]
    names  = cohorts_latin["name"]   if script=="latin" else cohorts_greek["name"]
    words  = cohorts_latin["word"]   if script=="latin" else cohorts_greek["word"]
    rand   = cohorts_latin["random"] if script=="latin" else cohorts_greek["random"]
    return dict(attested=attested, tokmatch=tokmatch, scrambled=scrambled,
                name=names, word=words, random=rand)

REPS = {}   # REPS[script][cohort] = [n, L+1, d]
for script in ["latin","greek"]:
    REPS[script] = {}
    cs = cohort_strings(script)
    for name, strs in cs.items():
        REPS[script][name] = build_matrix(strs)
    print(script, {k:v.shape for k,v in REPS[script].items()})
print("extraction done in", round(time.time()-t0,1), "s")



## Cell 8 — MATCH-QUALITY REPORT (printed *before* the headline)

Per the brief: *"this is the one I'd want to see before the headline, not after."* If `v_attested` and
`v_token_match` aren't actually matched on token-count and rarity, every downstream number is reading noise as
signal. We report achieved match on **token count** (exact) and **surprisal** (the rarity proxy).


In [ ]:
import numpy as np
from scipy import stats
a_nt = np.array([v["ntok_lat"] for v in VOCES]);  m_nt = np.array([v["tokmatch_ntok_lat"] for v in VOCES])
a_sp = np.array([v["surp_lat"] for v in VOCES]);   m_sp = np.array([v["tokmatch_surp_lat"] for v in VOCES])
print("=== TOKEN-COUNT match (attested vs token_match) ===")
print(f"  attested   tokens: mean {a_nt.mean():.2f}  sd {a_nt.std():.2f}  range [{a_nt.min()},{a_nt.max()}]")
print(f"  tokenmatch tokens: mean {m_nt.mean():.2f}  sd {m_nt.std():.2f}  range [{m_nt.min()},{m_nt.max()}]")
print(f"  per-pair |Δtokens|: mean {np.abs(a_nt-m_nt).mean():.2f}  (<=1 is good)")
print(f"  exact-count matches: {(a_nt==m_nt).sum()}/{len(a_nt)}")
print("=== SURPRISAL (rarity proxy) match ===")
print(f"  attested   surprisal: mean {a_sp.mean():.2f}  sd {a_sp.std():.2f}")
print(f"  tokenmatch surprisal: mean {m_sp.mean():.2f}  sd {m_sp.std():.2f}")
t,p = stats.ttest_rel(a_sp, m_sp)
print(f"  paired t-test attested vs match surprisal: t={t:.2f} p={p:.3f}  (p>0.05 = well matched)")
print(f"  per-pair |Δsurprisal|: mean {np.abs(a_sp-m_sp).mean():.2f}")
try:
    import matplotlib.pyplot as plt
    fig,ax=plt.subplots(1,2,figsize=(10,3.2))
    ax[0].hist(a_nt,alpha=.6,label="attested",bins=range(1,a_nt.max()+2))
    ax[0].hist(m_nt,alpha=.6,label="token_match",bins=range(1,a_nt.max()+2))
    ax[0].set_title("token count"); ax[0].legend()
    ax[1].hist(a_sp,alpha=.6,label="attested"); ax[1].hist(m_sp,alpha=.6,label="token_match")
    ax[1].set_title("surprisal"); ax[1].legend(); plt.tight_layout(); plt.show()
except Exception as e:
    print("(plot skipped:", e, ")")
print("\nMATCH-QUALITY VERDICT: read the two |Δ| means + the surprisal p-value above before trusting H1/H2.")



## Cell 9 — H1 Separability + Falsifier #4 (is it just surprisal?)

A logistic probe separates `v_attested` from `v_token_match` at each layer, **GroupKFold by family** so it
can't memorize individual strings (out-of-family accuracy is the real generalization test). Chance = 0.5.

**Falsifier #4** runs alongside: a probe using **surprisal alone** (1 feature). If surprisal-only matches the
residual-stream probe, "separability" was frequency in disguise. We report the gap flat.


In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline

def h1_layer_sweep(script):
    att = REPS[script]["attested"]; tm = REPS[script]["tokmatch"]
    fams = np.array([v["family"] for v in VOCES])
    groups = np.concatenate([fams, fams])           # token_match inherits its target's family
    y = np.array([1]*len(VOCES)+[0]*len(VOCES))
    n_groups = len(set(groups)); n_splits = min(5, n_groups)
    accs=[]
    for L in range(att.shape[1]):
        X = np.concatenate([att[:,L,:], tm[:,L,:]],0)
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.5))
        sc = cross_val_score(clf, X, y, groups=groups, cv=GroupKFold(n_splits), scoring="accuracy")
        accs.append(sc.mean())
    accs=np.array(accs); peak=int(accs.argmax())
    # Falsifier #4: surprisal-only probe
    sp = np.array([v["surp_lat" if script=="latin" else "surp_gr"] for v in VOCES]
                + [v["tokmatch_surp_lat" if script=="latin" else "tokmatch_surp_gr"] for v in VOCES])
    Xs = sp.reshape(-1,1)
    sp_acc = cross_val_score(make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000)),
                             Xs, y, groups=groups, cv=GroupKFold(n_splits), scoring="accuracy").mean()
    return accs, peak, sp_acc, n_splits

H1RES = {}
for script in ["latin","greek"]:
    accs, peak, sp_acc, ns = h1_layer_sweep(script)
    H1RES[script] = dict(peak_acc=float(accs[peak]), peak_layer=int(peak),
                         mean_acc=float(accs.mean()), surp_only_acc=float(sp_acc),
                         accs=[float(a) for a in accs])
    print(f"[{script}] H1 probe (GroupKFold x{ns} by family): peak acc {accs[peak]:.3f} @ layer {peak}"
          f" | mean-across-layers {accs.mean():.3f} | chance 0.500")
    print(f"[{script}] FALSIFIER#4 surprisal-only probe acc: {sp_acc:.3f}"
          f"  -> rep-probe beats surprisal by {accs[peak]-sp_acc:+.3f}")
    print(f"[{script}]   verdict: " + ("SEPARABILITY MAY BE SURPRISAL IN DISGUISE (gap small)"
          if accs[peak]-sp_acc < 0.05 else "rep-probe carries signal beyond surprisal"))
    try:
        import matplotlib.pyplot as plt
        plt.figure(figsize=(6,2.6)); plt.plot(accs,marker='.'); plt.axhline(0.5,ls='--',c='gray')
        plt.axhline(sp_acc,ls=':',c='red',label='surprisal-only')
        plt.title(f"H1 layer sweep ({script})"); plt.xlabel("layer"); plt.ylabel("acc"); plt.legend(); plt.show()
    except Exception as e: print("(plot skipped)", e)



## Cell 10 — H2 Name-likeness + Falsifier #1 (theonym split, T regressed) + the 2×2

**Theonym split first** (cheapest decisive cut, §5). For each vox at the peak/late layer, name-likeness =
`cos(vox, name_centroid) − cos(vox, random_centroid)`. We then:
1. compare **theonym-bearing vs pure-asemic** raw (Falsifier #1 raw form);
2. **regress out [surprisal, T]** and re-test on residuals — the decisive question is whether
   **pure-asemic, low-contamination** voces stay name-like *after* removing embedded god-names and familiarity;
3. repeat in **Greek script** → the 2×2 and Falsifier #2 (does the signature survive script migration).


In [ ]:
import numpy as np
from numpy.linalg import norm
from scipy import stats

def cos(a,b): return float(a@b/(norm(a)*norm(b)+1e-9))
def centroid(M_layer): return M_layer.mean(0)

def name_likeness(script, layer):
    name_c = centroid(REPS[script]["name"][:,layer,:])
    rand_c = centroid(REPS[script]["random"][:,layer,:])
    att = REPS[script]["attested"][:,layer,:]
    return np.array([cos(att[i], name_c) - cos(att[i], rand_c) for i in range(att.shape[0])])

def residualize(y, X):
    X1 = np.column_stack([np.ones(len(y)), X])
    beta,_,_,_ = np.linalg.lstsq(X1, y, rcond=None)
    return y - X1@beta

theo = np.array([v["theonym"] for v in VOCES])
T    = np.array([v["T"] for v in VOCES])

def compute_2x2(script, layer):
    surp = np.array([v["surp_lat" if script=="latin" else "surp_gr"] for v in VOCES])
    nl = name_likeness(script, layer)
    nb, pa = nl[theo], nl[~theo]
    _,pu = stats.mannwhitneyu(nb, pa, alternative="two-sided")
    nl_res = residualize(nl, np.column_stack([surp, T]))
    _,pu2 = stats.mannwhitneyu(nl_res[theo], nl_res[~theo], alternative="two-sided")
    low = (~theo) & (T < 0.15)
    t_pa,p_pa = stats.ttest_1samp(nl[low], 0.0) if low.sum()>1 else (np.nan, np.nan)
    return dict(nl=nl, nb=nb, pa=pa, pu=float(pu), pu_res=float(pu2),
                low_pa_mean=float(nl[low].mean()) if low.sum() else float("nan"),
                low_pa_p=float(p_pa), n_low_pa=int(low.sum()))

# ---- FULL LAYER SWEEP: the H1 sweep showed signal is layer-dependent (peaks early), so DON'T
#      trust a single hardcoded layer. Sweep the key Falsifier#1 quantity at every layer. ----
SWEEP = {s:[compute_2x2(s, L) for L in range(N_LAYERS+1)] for s in ["latin","greek"]}
print("pure-asemic LOW-T name-likeness by layer  (mean | vs-0 p ; * = sig & positive)")
print(f"{'layer':>5} | {'LATIN mean':>11} {'p':>7} | {'GREEK mean':>11} {'p':>7}")
for L in range(N_LAYERS+1):
    la,gr = SWEEP['latin'][L], SWEEP['greek'][L]
    sl = '*' if (la['low_pa_p']<0.05 and la['low_pa_mean']>0) else ' '
    sg = '*' if (gr['low_pa_p']<0.05 and gr['low_pa_mean']>0) else ' '
    print(f"{L:>5} | {la['low_pa_mean']:>+11.3f} {la['low_pa_p']:>6.3f}{sl} | {gr['low_pa_mean']:>+11.3f} {gr['low_pa_p']:>6.3f}{sg}")

NL_PEAK = {s:int(np.nanargmax([SWEEP[s][L]['low_pa_mean'] for L in range(N_LAYERS+1)])) for s in ['latin','greek']}
print("name-likeness peak layer (SELECTED post-hoc -> inflates significance, treat as exploratory):", NL_PEAK)
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(7,2.8))
    for s,c in [('latin','C0'),('greek','C1')]:
        plt.plot([SWEEP[s][L]['low_pa_mean'] for L in range(N_LAYERS+1)], marker='.', color=c, label=s)
    plt.axhline(0,ls='--',c='gray'); plt.legend(); plt.xlabel('layer')
    plt.ylabel('pure-asemic low-T\nname-likeness'); plt.title('H2 name-likeness layer sweep'); plt.show()
except Exception as e: print("(plot skipped)", e)

# Report the 2x2 at the H1-peak layer (where the model's distinction is strongest), with layer 2
# and 0.7N shown for contrast. results_2x2 (downstream) is fixed at the H1-peak layer.
REPORT_LAYER = H1RES['latin']['peak_layer']
LAYER = REPORT_LAYER  # alias for downstream cells
results_2x2 = {s:compute_2x2(s, REPORT_LAYER) for s in ['latin','greek']}
for L,tag in [(2,'layer-2 (early/form)'), (REPORT_LAYER,f'H1-peak L{REPORT_LAYER}'), (int(0.7*N_LAYERS),'0.7N (old default)')]:
    print(f"\n--- 2x2 @ {tag} ---")
    for script in ['latin','greek']:
        r=compute_2x2(script,L)
        verdict = "CARRIED-BY-THEONYMS" if (np.isnan(r['low_pa_p']) or r['low_pa_p']>0.05 or r['low_pa_mean']<=0) else "PURE-ASEMIC-STAYS-NAME-LIKE"
        print(f"  [{script:5}] theonym {r['nb'].mean():+.3f} | pure-asemic {r['pa'].mean():+.3f} | "
              f"low-T {r['low_pa_mean']:+.3f} (p={r['low_pa_p']:.3f},n={r['n_low_pa']}) | split p={r['pu']:.3f} | {verdict}")

# --- NAME-SPECIFICITY: the deflation check. Is it specifically NAME-like, or just generic real-lexical
#     surface? Surface adjacency would pull voces toward WORDS too. cos(vox,name)-cos(vox,word) > 0 means
#     specifically name-adjacent beyond generic-lexical. Run at the embedding floor (L0/L2) AND a deep layer. ---
def name_vs_word(script, layer):
    name_c = centroid(REPS[script]["name"][:,layer,:]); word_c = centroid(REPS[script]["word"][:,layer,:])
    att = REPS[script]["attested"][:,layer,:]
    return np.array([cos(att[i],name_c)-cos(att[i],word_c) for i in range(att.shape[0])])
low = (~theo) & (T<0.15)
print("\n--- NAME-SPECIFICITY  cos(vox,name)-cos(vox,word), pure-asemic low-T  (>0 = name-specific, not generic lexical) ---")
NAMESPEC={}
for script in ['latin','greek']:
    NAMESPEC[script]={}
    for L,tag in [(0,'embed L0'),(2,'early L2'),(int(0.7*N_LAYERS),'deep 0.7N')]:
        nvw = name_vs_word(script,L)[low]; t,p = stats.ttest_1samp(nvw,0.0)
        NAMESPEC[script][tag]=(float(nvw.mean()),float(p))
        print(f"  [{script:5} {tag:9}] name-minus-word {nvw.mean():+.3f} (p={p:.3f}) -> "
              + ("NAME-SPECIFIC" if (p<0.05 and nvw.mean()>0) else "NOT name-specific (generic lexical/surface)"))



In [ ]:
# --- the 2x2 summary table + Falsifier #2 (script survival) ---
print("THE 2x2  (name-likeness mean cos-diff; >0 = closer to names than random)")
print(f"{'':22}{'LATIN':>12}{'GREEK':>12}")
for grp,key in [("theonym-bearing","nb"),("pure-asemic","pa")]:
    lat = results_2x2['latin'][key].mean(); gr = results_2x2['greek'][key].mean()
    print(f"{grp:22}{lat:>12.3f}{gr:>12.3f}")
print(f"{'pure-asemic low-T':22}{results_2x2['latin']['low_pa_mean']:>12.3f}{results_2x2['greek']['low_pa_mean']:>12.3f}")
print()
lat_lp = results_2x2['latin']['low_pa_mean']; gr_lp = results_2x2['greek']['low_pa_mean']
print("FALSIFIER#2 (script migration): pure-asemic low-T name-likeness")
print(f"   Latin {lat_lp:+.3f}  ->  Greek {gr_lp:+.3f}")
print("   !! CIRCULARITY WARNING: 62/76 Greek strings are ALGORITHMIC Betz->Greek transliterations of the")
print("      Latin strings -- a deterministic re-encoding, NOT an independent attestation. So this headline")
print("      comparison is near-circular; do NOT read it as 'survives script migration'. The non-circular")
print("      test is the AUTHENTIC-GREEK arm (n=13 real PGM Greek) in Cell 10d, Test B3.")
# variance-tell: low SD across the pure-asemic set = shared ORTHOGRAPHIC feature, not distributed semantic
import numpy as np
for s in ['latin','greek']:
    nl=results_2x2[s]['nl']; low=( ~np.array([v['theonym'] for v in VOCES]) ) & (np.array([v['T'] for v in VOCES])<0.15)
    print(f"   variance-tell [{s}]: pure-asemic low-T name-likeness SD = {nl[low].std():.3f} "
          f"(tiny SD vs mean {nl[low].mean():+.3f} => uniform/orthographic, not distributed)")



## Cell 10d — TEST A (pure-asemic, real-not-theonyms) + TEST B (surface break)

Relayed from the design-side Claude, and they're right. Two tests answer different questions:

- **TEST A** — pure-asemic-*only* voces vs their token-matched pseudowords, **pre-registered layer** (H1-peak,
  declared before looking at the pure-asemic-only set), permutation null, GroupKFold by family. A clean result
  upgrades the lead to *"real surface effect, not theonyms"* — **not** to *"deep namehood"* (A's layer peaks
  early *because* that's where orthographic surface is most separable; pre-registering at H1-peak doesn't fully
  launder that).
- **TEST B** — re-run at the same layer under a genuine **surface break**: (B1) lowercase; (B3, critical)
  the **authentic-Greek** subset (real PGM Greek bytes, *not* transliterated — the only non-circular script
  arm). The sweep predicts B dies → confirms surface. If name-likeness *survives* a real surface break, the
  shallow reading is wrong and abstract-form gets its first honest evidence.

Framing fixed in advance so it's mis-sold in neither direction: **if A is real and B dies, the finding is
"the model recognizes the orthographic signature of a name-of-power on sight, before contextualization"** —
surface-recognition-of-namehood, real and interesting, but not deep operative form.


In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline

PREREG_LAYER = H1RES['latin']['peak_layer']   # pre-registered BEFORE looking at the pure-asemic-only set
pa_idx  = np.where(~np.array([v['theonym'] for v in VOCES]))[0]
pa_fams = np.array([VOCES[i]['family'] for i in pa_idx])

def probe_acc(att_reps, ctrl_reps, fams, layer, n_perm=0):
    X = np.concatenate([att_reps[:,layer,:], ctrl_reps[:,layer,:]],0)
    y = np.array([1]*len(att_reps)+[0]*len(att_reps)); g = np.concatenate([fams,fams])
    ns = min(5, len(set(g)))
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.5))
    cv = GroupKFold(ns)
    # error_score=nan + nanmean: tiny-n cohorts (authentic-Greek n=13) can yield a single-class fold under
    # label permutation; score it nan and continue instead of emitting a scary FitFailedWarning/raising.
    acc = np.nanmean(cross_val_score(clf,X,y,groups=g,cv=cv,scoring='accuracy',error_score=np.nan))
    p=None
    if n_perm:
        rng=np.random.RandomState(0); cnt=0; valid=0
        for _ in range(n_perm):
            a=np.nanmean(cross_val_score(clf,X,rng.permutation(y),groups=g,cv=cv,scoring='accuracy',error_score=np.nan))
            if not np.isnan(a): valid+=1; cnt += (a>=acc)
        p=(cnt+1)/(valid+1) if valid else float('nan')
    return acc,p

def name_like(reps_att, script, layer):
    nc=REPS[script]['name'][:,layer,:].mean(0); rc=REPS[script]['random'][:,layer,:].mean(0)
    A=reps_att[:,layer,:]
    return np.array([float(A[i]@nc/(np.linalg.norm(A[i])*np.linalg.norm(nc)+1e-9)
                          - A[i]@rc/(np.linalg.norm(A[i])*np.linalg.norm(rc)+1e-9)) for i in range(len(A))])

# ===== TEST A: pure-asemic separability, latin, pre-registered layer =====
attA = REPS['latin']['attested'][pa_idx]; ctrlA = REPS['latin']['tokmatch'][pa_idx]
accA,pA = probe_acc(attA,ctrlA,pa_fams,PREREG_LAYER,n_perm=200)
sweepA = [probe_acc(attA,ctrlA,pa_fams,L)[0] for L in range(N_LAYERS+1)]
print(f"TEST A [pure-asemic only, n={len(pa_idx)}] separability @ PRE-REG layer {PREREG_LAYER}: "
      f"acc {accA:.3f}, perm-p {pA:.3f}")
print(f"  secondary sweep peak {max(sweepA):.3f} @ L{int(np.argmax(sweepA))} (peak-pick inflates; Bonferroni x{N_LAYERS+1})")
print(f"  -> {'REAL (not theonyms): pure-asemic voces separable from matched nonsense' if pA<0.05 else 'NOT separable -> H1 was carried by theonyms'}")



In [ ]:
# ===== TEST B1: LOWERCASE surface break (re-extract, same pre-reg layer) =====
pa_low   = [VOCES[i]['latin'].lower()        for i in pa_idx]
ctrl_low = [VOCES[i]['tokmatch_lat'].lower() for i in pa_idx]
attB  = np.stack([extract_reps(s)[0] for s in pa_low])
ctrlB = np.stack([extract_reps(s)[0] for s in ctrl_low])
accB,pB = probe_acc(attB,ctrlB,pa_fams,PREREG_LAYER,n_perm=200)
print(f"TEST B1 [lowercase] separability @ L{PREREG_LAYER}: acc {accB:.3f} (perm-p {pB:.3f}) "
      f"| Δ vs UPPERCASE {accB-accA:+.3f}")
print(f"  -> {'SURVIVES surface break (acc holds) -> NOT pure-orthographic' if (accB-accA)>-0.10 and pB<0.05 else 'DIES under lowercase -> SURFACE/orthographic (capitalization-shaped name signature)'}")

# ===== TEST B3 (critical, non-circular): AUTHENTIC-GREEK arm, real PGM Greek bytes =====
auth_idx = np.array([i for i,v in enumerate(VOCES) if 'attested_greek' in v['tags'] and not v['theonym']])
if len(auth_idx) >= 6:
    af = np.array([VOCES[i]['family'] for i in auth_idx])
    attG = REPS['greek']['attested'][auth_idx]; ctrlG = REPS['greek']['tokmatch'][auth_idx]
    accG,pG = probe_acc(attG,ctrlG,af,PREREG_LAYER,n_perm=200)
    nlG = name_like(attG,'greek',PREREG_LAYER)
    from scipy import stats as _st; tG,pnlG = _st.ttest_1samp(nlG,0.0)
    print(f"TEST B3 [AUTHENTIC Greek, n={len(auth_idx)} real PGM bytes] @ L{PREREG_LAYER}: "
          f"separability acc {accG:.3f} (perm-p {pG:.3f}) | name-likeness {nlG.mean():+.3f} (p={pnlG:.3f})")
    print(f"  -> non-circular script test: "
          f"{'real PGM-Greek voces ARE special/name-like (abstract form gets honest support)' if (pG<0.05) else 'authentic Greek NOT separable -> the Greek headline was the transliteration circularity'}")
else:
    print("TEST B3 skipped: too few authentic-Greek pure-asemic voces.")



## Cell 10e — THE split: authentic-Greek vs transliterated-Greek (the deciding plot)

The fp16 run surfaced the one result that doesn't fit "surface, decays": **Greek name-likeness persists deep**
while Latin dies by mid-network. But that persistence lands *exactly* on the transliteration confound — 62/76
Greek strings are algorithmic Betz→Greek re-encodings of the Latin. So the question that decides which paper
this is: **does the persistence hold for the 13 *authentic* PGM-Greek strings alone, or only for the
transliterations?**

- **Authentic Greek persists deep too** → it is not a transliteration artifact; the abstraction reading gets
  its first real evidence.
- **Authentic Greek decays like Latin, only transliterated-Greek persists** → deep-Greek persistence *was* a
  transliteration artifact; shallow/surface wins cleanly.

Two views: (1) the split-sweep (authentic-13 vs transliterated vs Latin) — confounds string-set, so also
(2) a **same-strings control**: the very same 13 strings encoded authentic-Greek vs algorithmic-translit,
which isolates *encoding* from *which strings*. n=13 is small → exploratory; read the **shape**, not decimals.


In [ ]:
import numpy as np, matplotlib.pyplot as plt
theo = np.array([v['theonym'] for v in VOCES]); T = np.array([v['T'] for v in VOCES])
auth_mask  = np.array(['attested_greek' in v['tags'] and not v['theonym'] for v in VOCES])
trans_mask = (~theo) & (T<0.15) & (~auth_mask)            # transliterated, pure-asemic, low-T
print(f"authentic-Greek pure-asemic n={int(auth_mask.sum())} | transliterated-Greek pure-asemic low-T n={int(trans_mask.sum())}")

def _nl(A, nc, rc, i):
    a=A[i]; return float(a@nc/(np.linalg.norm(a)*np.linalg.norm(nc)+1e-9) - a@rc/(np.linalg.norm(a)*np.linalg.norm(rc)+1e-9))
def nl_sweep(script, mask_or_reps):
    reps = REPS[script]['attested'][mask_or_reps] if isinstance(mask_or_reps, np.ndarray) and mask_or_reps.dtype==bool else mask_or_reps
    out=[]
    for L in range(N_LAYERS+1):
        nc=REPS[script]['name'][:,L,:].mean(0); rc=REPS[script]['random'][:,L,:].mean(0); A=reps[:,L,:]
        out.append(np.mean([_nl(A,nc,rc,i) for i in range(len(A))]))
    return np.array(out)

auth_g  = nl_sweep('greek', auth_mask)
trans_g = nl_sweep('greek', trans_mask)
lat_ref = nl_sweep('latin', trans_mask)        # same translit strings in Latin = the 'surface decays' reference
DEEP = slice(int(0.5*N_LAYERS), N_LAYERS+1)
print(f"\nDEEP-layer (L{int(0.5*N_LAYERS)}-{N_LAYERS}) mean name-likeness:")
print(f"  authentic-Greek    : {auth_g[DEEP].mean():+.3f}")
print(f"  transliterated-Grk : {trans_g[DEEP].mean():+.3f}")
print(f"  Latin (reference)  : {lat_ref[DEEP].mean():+.3f}")

# same-strings de-confound: the 13 authentic strings, re-encoded as algorithmic transliteration
auth_idx = np.where(auth_mask)[0]
algo_reps = np.stack([extract_reps(to_greek(VOCES[i]['latin']))[0] for i in auth_idx])
algo_g = nl_sweep('greek', algo_reps)
print(f"\nSAME-STRINGS control (the {len(auth_idx)} authentic strings, two encodings), DEEP mean:")
print(f"  authentic-Greek bytes : {auth_g[DEEP].mean():+.3f}")
print(f"  algorithmic translit  : {algo_g[DEEP].mean():+.3f}")
print("  -> " + ("encoding-INVARIANT (transliteration is representationally faithful)"
        if abs(auth_g[DEEP].mean()-algo_g[DEEP].mean())<0.02 else
        "ENCODING MATTERS (authentic != algorithmic for the SAME strings)"))

VERDICT = ("AUTHENTIC GREEK PERSISTS DEEP -> NOT a transliteration artifact; abstraction reading gets first real evidence"
           if auth_g[DEEP].mean() > 0.02 else
           "AUTHENTIC GREEK DECAYS like Latin -> deep-Greek persistence was a TRANSLITERATION ARTIFACT; SHALLOW wins cleanly")
print("\n>>> WHICH-PAPER VERDICT:", VERDICT)

plt.figure(figsize=(7.6,3.1))
plt.plot(auth_g, marker='o', label=f'authentic Greek (n={int(auth_mask.sum())})', color='C2')
plt.plot(trans_g, marker='.', label=f'transliterated Greek (n={int(trans_mask.sum())})', color='C1')
plt.plot(lat_ref, marker='.', ls='--', label='Latin (same translit strings)', color='C0')
plt.plot(algo_g, marker='x', ls=':', label=f'authentic strings, ALGO-translit (n={len(auth_idx)})', color='C3')
plt.axhline(0,ls=':',c='gray'); plt.legend(fontsize=7.5); plt.xlabel('layer'); plt.ylabel('name-likeness')
plt.title('THE split: does authentic Greek persist deep, or decay like Latin?'); plt.tight_layout(); plt.show()

# stash for JSON
GREEK_SPLIT = dict(authentic_deep=float(auth_g[DEEP].mean()), translit_deep=float(trans_g[DEEP].mean()),
                   latin_deep=float(lat_ref[DEEP].mean()), algo_deep=float(algo_g[DEEP].mean()),
                   n_authentic=int(auth_mask.sum()), n_translit=int(trans_mask.sum()),
                   authentic_sweep=[float(x) for x in auth_g], translit_sweep=[float(x) for x in trans_g])



## Cell 10f — THE DECIDER: is deep-Greek persistence voces-specific, or just Greek script?

The Cell 10e split killed the *transliteration* confound (authentic Greek persists like transliterated). But its
auto-banner overclaimed: it only ever compared voces-in-Greek to the *Latin* baseline — never to **controls in
Greek**. This cell closes that: render the token-matched nonsense controls in Greek too, and ask whether the
voces persist deeper than their *own matched Greek nonsense*. If not, "Greek persists" is a fact about Greek
**script**, not about the voces — and the abstraction reading is dissolved (not by argument, by control).


In [ ]:
import numpy as np
from scipy import stats
def _nlvals(reps, script, L):
    nc=REPS[script]['name'][:,L,:].mean(0); rc=REPS[script]['random'][:,L,:].mean(0); A=reps[:,L,:]
    nrm=lambda x: x/(np.linalg.norm(x)+1e-9)
    return np.array([float(nrm(A[i])@nrm(nc) - nrm(A[i])@nrm(rc)) for i in range(len(A))])
theo=np.array([v['theonym'] for v in VOCES]); T=np.array([v['T'] for v in VOCES]); low=(~theo)&(T<0.15)
DEEP=range(int(0.5*N_LAYERS), N_LAYERS+1)
VOCES_SPECIFICITY={}
for script in ['greek','latin']:
    voxv=np.mean([_nlvals(REPS[script]['attested'][low],script,L) for L in DEEP],axis=0)
    ctlv=np.mean([_nlvals(REPS[script]['tokmatch'][low],script,L) for L in DEEP],axis=0)
    _,p=stats.mannwhitneyu(voxv,ctlv,alternative='greater')
    VOCES_SPECIFICITY[script]=dict(vox_deep=float(voxv.mean()), ctl_deep=float(ctlv.mean()), p=float(p), n=int(low.sum()))
    print(f"[{script}] DEEP name-likeness: voces {voxv.mean():+.3f}  vs  token-match controls {ctlv.mean():+.3f}  | voces>ctl p={p:.3f}")
    print("   ->", "VOCES-SPECIFIC (abstraction reading gets real support)" if p<0.05
          else "NOT voces-specific -> it's GREEK SCRIPT, not voces-texture")
print("\nRETRACTION: the Cell 10e banner ('abstraction reading gets first real evidence') is SUPERSEDED by this cell.")
print("Headline = surface texture-recognition (H1, shallow); deep-Greek persistence is a Greek-SCRIPT effect, not voces.")



## Cell 11 — Verdict board: the four falsifiers, flat

Read this with the brief's caution in mind: *the wanting is a confound in the reader, not just the model.*
Boring/negative outcomes are reported with the same weight as positive ones.


In [ ]:
print("="*70)
print("VERDICT BOARD  —  model:", MODEL_NAME, "| tier:", TIER)
print("="*70)
# F4 from cell 9 (recompute peak gap, latin)
accs,peak,sp_acc,_ = h1_layer_sweep("latin")
print(f"[F4] Separability = surprisal?   rep-probe peak {accs[peak]:.3f} vs surprisal-only {sp_acc:.3f}")
print("     ->", "KILLED headline (gap<0.05): separability ~ surprisal" if accs[peak]-sp_acc<0.05
      else "survives: rep-probe carries signal beyond frequency")
la=results_2x2['latin']
print(f"[F1] Name-likeness = theonyms only?  pure-asemic low-T mean {la['low_pa_mean']:+.3f} (p={la['low_pa_p']:.3f})")
print("     ->", "KILLED strong claim: pure-asemic not name-like" if (np.isnan(la['low_pa_p']) or la['low_pa_p']>0.05 or la['low_pa_mean']<=0)
      else "survives: pure-asemic voces are name-like net of T")
print(f"[F2] Dies under Greek script?   Latin {results_2x2['latin']['low_pa_mean']:+.3f} -> Greek {results_2x2['greek']['low_pa_mean']:+.3f}")
print("     ->", "KILLED: typographic string-cache" if not(results_2x2['greek']['low_pa_mean']>0 and results_2x2['latin']['low_pa_mean']>0)
      else "survives: abstract form, not byte-string")
print("[F3] Steering control vector — NOT TESTED (H4 out of MVP scope).")
print("-"*70)
print("[caveat-0] SCALE: this is", TIER, "- a",
      "pilot" if "1.5B" in MODEL_NAME else "sub-scale run", "not the >=7B-at-power headline.")
print("[caveat-2] QUANT:", ("4-BIT -> any headline carries 'may be sensitive to quantization';"
      " fp16 confirmation pass needed." if LOAD_4BIT else
      "fp16, un-quantized -> quantization caveat does NOT apply to this run."))



## Cell 12 — Persist results + breakage log

Saves a JSON the human can paste back up the chain, plus a free-text **breakage log** — the cell that broke,
the cohort that wouldn't build cleanly, anything improvised off-spec. *The failures carry more information than
the successes.*


In [ ]:
import json, numpy as np
def jsonable(x):
    if isinstance(x,(np.floating,)): return float(x)
    if isinstance(x,(np.integer,)): return int(x)
    if isinstance(x,np.ndarray): return x.tolist()
    return x
summary = dict(
    model=MODEL_NAME, tier=TIER, quantized=bool(LOAD_4BIT), n_layers=int(N_LAYERS), layer_used=int(LAYER), seed=SEED,
    n_voces=len(VOCES), n_theonym=int(sum(v['theonym'] for v in VOCES)),
    match_quality=dict(
        dtok_mean=float(np.abs(np.array([v['ntok_lat'] for v in VOCES])-np.array([v['tokmatch_ntok_lat'] for v in VOCES])).mean()),
        dsurp_mean=float(np.abs(np.array([v['surp_lat'] for v in VOCES])-np.array([v['tokmatch_surp_lat'] for v in VOCES])).mean()),
    ),
    report_layer=int(REPORT_LAYER),
    h1={s:{k:H1RES[s][k] for k in ['peak_acc','peak_layer','mean_acc','surp_only_acc']} for s in ['latin','greek']},
    twoxtwo={s:{k:jsonable(np.mean(results_2x2[s][k])) for k in ['nb','pa']} |
                {'low_pa_mean':jsonable(results_2x2[s]['low_pa_mean']),
                 'n_low_pa':int(results_2x2[s]['n_low_pa']),
                 'raw_split_p':jsonable(results_2x2[s]['pu']),
                 'resid_split_p':jsonable(results_2x2[s]['pu_res']),
                 'low_pa_vs0_p':jsonable(results_2x2[s]['low_pa_p'])} for s in ['latin','greek']},
    sweep_low_pa_mean={s:[jsonable(SWEEP[s][L]['low_pa_mean']) for L in range(N_LAYERS+1)] for s in ['latin','greek']},
    nl_peak_layer={s:int(v) for s,v in NL_PEAK.items()},
    greek_split=GREEK_SPLIT if 'GREEK_SPLIT' in dir() else None,
    voces_specificity=VOCES_SPECIFICITY if 'VOCES_SPECIFICITY' in dir() else None,
)
summary["version"] = VERSION
RESULTS_FILE = f"voces_{VERSION}_results.json"
with open(RESULTS_FILE,"w") as f: json.dump(summary,f,indent=2)
print(json.dumps(summary,indent=2))

dtokm = summary['match_quality']['dtok_mean']; dsurpm = summary['match_quality']['dsurp_mean']
BREAKAGE_LOG = f'''
=== BREAKAGE / SURPRISE LOG (auto-filled; edit/add by hand) ===
- Model / tier: {MODEL_NAME} | {TIER} | quantized={LOAD_4BIT}
- Token-match convergence (Cell 8): |Δtok| mean {dtokm:.2f} (38/38-style exact match is ideal);
    |Δsurprisal| mean {dsurpm:.2f}. Note: a small paired-t p<0.05 on surprisal means controls run
    slightly rarer than attested -> a CONSERVATIVE confound (works against H1, not for it).
- Cohorts: all built. (Flag here if any cohort failed.)
- H1 separability: latin peak {H1RES['latin']['peak_acc']:.3f} @ layer {H1RES['latin']['peak_layer']},
    greek peak {H1RES['greek']['peak_acc']:.3f} @ layer {H1RES['greek']['peak_layer']}.
    Surprisal-only: latin {H1RES['latin']['surp_only_acc']:.3f}, greek {H1RES['greek']['surp_only_acc']:.3f}
    -> Falsifier#4 {'SURVIVED (rep-probe >> surprisal)' if H1RES['latin']['peak_acc']-H1RES['latin']['surp_only_acc']>0.05 else 'AT RISK'}.
- Layer where signal peaks: H1 peaks EARLY (~layer 2), not at the old 0.7N default ({int(0.7*N_LAYERS)}).
    The original single-layer name-likeness was likely read at the wrong (decayed) layer.
- H2 name-likeness peak layer (post-hoc selected): {NL_PEAK}
- Surprise: {'Latin vs Greek name-likeness DIVERGE' } -- examine the sweep table/plot above, not one layer.
- Off-spec improvisations: surprisal used as token-rarity proxy (no external freq table); Greek script is
    algorithmic Betz->Greek transliteration; corpus curated from PGM knowledge (see Digital Ambler sources
    for provenance tightening), not parsed from a verified Betz edition.
'''
print(BREAKAGE_LOG)
BREAKAGE_FILE = f"voces_{VERSION}_breakage.txt"
open(BREAKAGE_FILE,"w").write(BREAKAGE_LOG)

# freeze the EXACT model-generated controls (archived-exactly > merely regenerable).
# Each entry is a SELF-CONTAINED match-pair: target + control strings AND both their
# token-counts/surprisals AND the deltas -> an auditor verifies the token-isomorphism
# from this file alone, without rerunning extraction.
def _r(x): return round(float(x), 4) if x is not None else None
FROZEN = dict(model=MODEL_NAME, seed=SEED, version=VERSION,
    schema="each pair: dtok=target_ntok-tokmatch_ntok (should be ~0), dsurp=target_surp-tokmatch_surp",
    token_matched_pairs=[dict(
        target=v["latin"], target_ntok=v["ntok_lat"], target_surp=_r(v["surp_lat"]),
        tokmatch_latin=v["tokmatch_lat"], tokmatch_ntok=v.get("tokmatch_ntok_lat"),
        tokmatch_surp=_r(v.get("tokmatch_surp_lat")),
        dtok=(v["ntok_lat"] - v["tokmatch_ntok_lat"]) if v.get("tokmatch_ntok_lat") is not None else None,
        dsurp=_r((v["surp_lat"] - v["tokmatch_surp_lat"]) if v.get("tokmatch_surp_lat") is not None else None),
        tokmatch_greek=v["tokmatch_gr"], scrambled_latin=v["scrambled_lat"])
        for v in VOCES],
    anchors=dict(name=cohorts_latin["name"], word=cohorts_latin["word"], random=cohorts_latin["random"]))
FROZEN_FILE = f"voces_{VERSION}_frozen_controls.json"
open(FROZEN_FILE,"w").write(json.dumps(FROZEN, ensure_ascii=False, indent=2))
print(f"saved: {RESULTS_FILE}, {BREAKAGE_FILE}, {FROZEN_FILE}  (download all from the Files pane)")

